# Notebook 3 — Anomaly Detection
Find unusual financial patterns using Z-score and Isolation Forest. Flags are exported to CSV for loading into `fact_anomaly_flags`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

DATA = Path('../data/clean')
pl = pd.read_csv(DATA / 'profitandloss.csv')
bs = pd.read_csv(DATA / 'balancesheet.csv')
cf = pd.read_csv(DATA / 'cashflow.csv')

pl = pl[pl['is_ttm'] == False].copy()
bs = bs[bs['is_ttm'] == False].copy()
cf = cf[cf['is_ttm'] == False].copy()
print('Data loaded.')

## 1 — Z-score anomaly detection per metric per company

In [ ]:
ZSCORE_THRESHOLD = 2.5
z_records = []

for sym, grp in pl.groupby('company_id'):
    for col in ['sales', 'net_profit', 'operating_profit']:
        vals = grp[col].dropna()
        if len(vals) < 3:
            continue
        z = (vals - vals.mean()) / (vals.std() + 1e-9)
        flagged = grp.loc[z.abs() >= ZSCORE_THRESHOLD, ['company_id','fiscal_year']]
        for _, row in flagged.iterrows():
            z_val = z.loc[row.name]
            z_records.append({'symbol': sym, 'fiscal_year': row['fiscal_year'],
                              'metric': col, 'z_score': round(z_val, 3),
                              'direction': 'spike' if z_val > 0 else 'drop',
                              'method': 'zscore'})

for sym, grp in bs.groupby('company_id'):
    vals = grp['borrowings'].dropna()
    if len(vals) < 3:
        continue
    z = (vals - vals.mean()) / (vals.std() + 1e-9)
    flagged = grp.loc[z.abs() >= ZSCORE_THRESHOLD, ['company_id','fiscal_year']]
    for _, row in flagged.iterrows():
        z_val = z.loc[row.name]
        z_records.append({'symbol': sym, 'fiscal_year': row['fiscal_year'],
                          'metric': 'borrowings', 'z_score': round(z_val, 3),
                          'direction': 'spike' if z_val > 0 else 'drop',
                          'method': 'zscore'})

z_flags = pd.DataFrame(z_records)
print(f'Z-score anomalies detected: {len(z_flags)}')
print(z_flags.groupby('metric')['symbol'].count())

## 2 — Isolation Forest on the full feature matrix

In [ ]:
feat_pl = pl.groupby('company_id')[['sales','net_profit','operating_profit','interest','eps']].mean()
feat_bs = bs.groupby('company_id')[['borrowings','total_assets','debt_to_equity']].mean()
feat_cf = cf.groupby('company_id')[['operating_activity','free_cash_flow']].mean()
feat = feat_pl.join(feat_bs, how='outer').join(feat_cf, how='outer').fillna(0)

scaler = StandardScaler()
X = scaler.fit_transform(feat)

iso = IsolationForest(contamination=0.1, random_state=42)
feat['iso_flag'] = iso.fit_predict(X)
feat['iso_score'] = iso.score_samples(X)

iso_anomalies = feat[feat['iso_flag'] == -1].index.tolist()
print(f'Isolation Forest anomalies: {len(iso_anomalies)}')
print('Companies flagged:', iso_anomalies[:20])

## 3 — Compare Z-score vs Isolation Forest results

In [ ]:
z_companies = set(z_flags['symbol'].unique())
iso_companies = set(iso_anomalies)
overlap = z_companies & iso_companies

fig, ax = plt.subplots(figsize=(8, 4))
counts = [len(z_companies), len(iso_companies), len(overlap)]
bars = ax.bar(['Z-score only', 'Isolation Forest only', 'Both methods'], counts,
              color=['steelblue','salmon','#10b981'], alpha=0.85)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_title('Anomaly detection method comparison')
ax.set_ylabel('Companies flagged')
plt.tight_layout()
plt.show()
print(f'Flagged by both methods (high confidence): {sorted(overlap)}')

## 4 — Anomaly timeline: which companies had the most unusual years?

In [ ]:
top_anomaly_cos = z_flags.groupby('symbol').size().nlargest(8).index.tolist()

fig, axes = plt.subplots(len(top_anomaly_cos), 1, figsize=(14, 3*len(top_anomaly_cos)))
if len(top_anomaly_cos) == 1:
    axes = [axes]

for ax, sym in zip(axes, top_anomaly_cos):
    grp = pl[pl['company_id'] == sym].sort_values('fiscal_year')
    ax.plot(grp['fiscal_year'], grp['sales'], marker='o', label='Sales', color='steelblue')
    ax2 = ax.twinx()
    ax2.plot(grp['fiscal_year'], grp['net_profit'], marker='s', label='Net profit',
             color='green', linestyle='--', alpha=0.7)
    flagged_yrs = z_flags[(z_flags['symbol'] == sym) & (z_flags['metric'] == 'sales')]['fiscal_year'].values
    for yr in flagged_yrs:
        ax.axvline(yr, color='red', linewidth=1.5, linestyle=':', alpha=0.8)
    ax.set_title(f'{sym} — sales trend (red dotted = anomaly year)')
    ax.set_ylabel('Sales (₹ Cr)', color='steelblue')
    ax2.set_ylabel('Net profit (₹ Cr)', color='green')

plt.tight_layout()
plt.show()

## 5 — Isolation Forest anomaly score scatter

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
feat_sorted = feat.sort_values('iso_score')
colors_iso = ['#ef4444' if f == -1 else 'steelblue' for f in feat_sorted['iso_flag']]
ax.bar(range(len(feat_sorted)), feat_sorted['iso_score'], color=colors_iso, alpha=0.85, width=0.8)
ax.set_xticks(range(len(feat_sorted)))
ax.set_xticklabels(feat_sorted.index, rotation=90, fontsize=7)
ax.axhline(feat[feat['iso_flag']==-1]['iso_score'].max(), color='red', linestyle='--', alpha=0.6)
ax.set_title('Isolation Forest anomaly scores (red = flagged)')
ax.set_ylabel('Anomaly score (lower = more anomalous)')
plt.tight_layout()
plt.show()

## 6 — Export anomaly flags to CSV

In [ ]:
iso_df = pd.DataFrame({
    'symbol': iso_anomalies,
    'fiscal_year': None,
    'metric': 'multi-metric',
    'z_score': None,
    'direction': 'anomaly',
    'method': 'isolation_forest'
})

all_flags = pd.concat([z_flags, iso_df], ignore_index=True)
all_flags['flagged_at'] = pd.Timestamp.utcnow().date()
all_flags.to_csv('../data/clean/anomaly_flags.csv', index=False)
print(f'Exported {len(all_flags)} anomaly flags to data/clean/anomaly_flags.csv')